# Tutorial 2 (local model): an agent that drives the simulator, with an LLM on your own computer

In the first tutorial **you** built a model, ran it and looked at the results. Here a program does those steps for you: you
give it a request in plain English, and it decides which `kusim` commands to run, reads the answers, fixes mistakes and
reports. The "brain" is a large language model (LLM). In this version it runs **on your computer** through [Ollama](https://ollama.com),
so there is no account, no key, no quota, and nothing you type or compute leaves the machine.

By the end you will have:

1. built the **loop** (observe, act, check) that makes a program an *agent*, and run it on three tasks;
2. seen what a **small local model** (8 billion parameters) can and cannot do, and why you audit it;
3. **audited** what the agent did, and caught it claiming more than the data support.

**What you need.**
- The kusim package and the Python packages of tutorial 1.
- Ollama installed and running, and the model `qwen3:8b` (about 5 GB; `ollama pull qwen3:8b`). It needs roughly 8 GB of free memory; a GPU makes it much faster.
- Patience: a local model needs **minutes per task**, not seconds. A hosted "frontier" model is faster and more reliable; see `tutorial_agentic_workflow_frontier.ipynb`.

If Ollama or the model is missing, the notebook says so and falls back to a *scripted* model that makes the same tool calls a careful model makes, so everything
still runs offline and gives the same result every time.

**How an agent works.** It is a loop that repeats until the model gives an answer:

1. You give a task. The program sends the model the task, the list of tools and everything said so far. The model only ever sees text.
2. The model replies with either a final answer or a **tool call**, for example "call `run` with `model=model.json`".
3. The program checks the call, asks **you** first if it is expensive, runs `kusim run ...` and reads its JSON answer.
4. The answer, shortened, is added to the conversation, and the loop goes back to step 2.

The model never touches your files. It can only ask; the program decides what is allowed.

## 0. Set up

The helper modules are in this folder: `agent_lab.py` (the connection, the tools, the loop), `agent_lab_case.py` (the files the tasks work on) and `agent_lab_scripts.py` (the three requests and the scripted plans).

In [ ]:
import importlib.util
import json
import os
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

HERE = Path.cwd() if Path("agent_lab.py").exists() else Path("examples")      # this notebook's folder
sys.path.insert(0, str(HERE.resolve()))
import agent_lab as al
import agent_lab_case as case
import agent_lab_scripts as sc

import kusim
print("kusim", kusim.__version__)

In [ ]:
# a fresh folder for the agent to work in (the agent can not leave it), and the "hidden truth" next to it
shutil.rmtree(HERE / "output" / "agent_lab_local", ignore_errors=True)
folders = case.make_workspace(HERE / "output" / "agent_lab_local")
WORK = folders["work"]
for path in sorted(WORK.rglob("*")):
    if path.is_file():
        print(path.relative_to(WORK).as_posix())

`model.json` is for Task A, `model_b.json` for Task B and the folder `field/` for Task C: a deck, measured data with the field's well
names, and the description of what is uncertain. The real permeability that produced those data is kept in `hidden/`; the agent
has no tool that can read it, and we use it only at the end to check the agent's claims.

## 1. The tools: the command line, described to a model

A model can only use what it is told about. kusim was built so that every capability is a command with documented options and
one JSON answer, so the tool descriptions can be **generated** from the command line itself (`kusim.cli.build_parser()`). Add a command or
an option to kusim and the agent gets it.

In [ ]:
tools, commands = al.kusim_tools()
pd.DataFrame([(t["name"], t["description"][:95]) for t in tools], columns=["tool", "what the model is told"])

In [ ]:
# what the model receives for one tool: a name, a description and a JSON Schema of the arguments
edit = next(t for t in tools if t["name"] == "edit")
print(json.dumps(edit, indent=1)[:1500])

Three things are *not* in the list on purpose: the guardrail limits (`max_cells`, `max_seconds`, ... an agent may lower them, not
raise them), editing a model in place (an agent writes a new file), and a way out of the workspace folder. `ensemble` has caps
(at most 4 workers and 100 members here) and **waits for your approval**, because it runs many simulations.

## 2. Prepare the local model

Open a terminal once and run `ollama pull qwen3:8b` (a download of about 5 GB). Ollama must be running (the desktop app, or `ollama serve`).

**Why a special copy of the model.** Ollama gives every model a *context window* of only 4096 tokens by default. An agent's conversation (the tool
list, the instructions and every answer so far) is longer than that, so the model silently forgets the start of it, and may answer nothing at all. `al.ollama_prepare` makes
a copy called `qwen3-16k` with a window of 16384 tokens (a few seconds, no download; the copy takes no extra disk space worth mentioning).

**What a small model is like.** It picks tools and fills in arguments less reliably than a hosted model, it sometimes invents options (a file that does not exist, an argument
that is not there), and it thinks for a long time before each call. The agent's checks send every bad call back with the reason, so it usually recovers. Read every answer critically.

In [ ]:
MODEL_BASE, MODEL = "qwen3:8b", "qwen3-16k"
print("models installed in Ollama:", al.ollama_models())
try:
    MODEL = al.ollama_prepare(MODEL_BASE, MODEL, num_ctx=16384)
    LIVE = True
    print("ready:", MODEL)
except al.LLMError as problem:
    LIVE = False
    print(problem)
    print("\n-> continuing with the scripted model: the same tool calls, no local model needed.")


def new_agent(task, **kwargs):
    """An agent for 'A', 'B' or 'C': the local model if it is ready, else the scripted plan of that task."""
    llm = al.connect("ollama", model=MODEL) if LIVE else al.connect("scripted", script=sc.SCRIPTS[task])
    (WORK / f"trace_{task}.jsonl").unlink(missing_ok=True)
    return al.Agent(llm, WORK, approve=al.ask_user if LIVE else al.auto_approve, trace=f"trace_{task}.jsonl", **kwargs)

In [ ]:
# one test call: does the model ask for a tool when we ask it to? (the first call also loads the model into memory: it can take a minute)
import time
if LIVE:
    llm = al.connect("ollama", model=MODEL)
    started = time.time()
    answer = llm.chat([{"role": "user", "content": "Call the tool list_files once, with no arguments."}],
                      [t for t in tools if t["name"] == "list_files"])
    print(f"the model asked for: {[(c.name, c.arguments) for c in answer.tool_calls] or answer.text}  ({time.time() - started:.0f} s)")
else:
    print("(skipped: no local model)")

**Using another local model.** Change `MODEL_BASE` to any model you have pulled that supports **tools** (for example `llama3.1:8b` or `mistral-nemo`) and
`MODEL` to a name for its large-window copy, then run the cell again. Not every model supports tools: in our own tests `llama3.2` (3B) used them wrongly and `mistral` (7B) did not use them at all,
while `qwen3:8b` completed the first two tasks. Section 9 compares models. To use a **hosted** model instead, see `tutorial_agentic_workflow_frontier.ipynb`.

## 3. You are the agent first

Before the loop, do what the loop does, by hand. `helper.call(tool, **arguments)` makes one tool call with the same checks the agent's calls get,
and returns the answer as a dict. Answers are long, so the agent sees a shortened form (`al.compact`): long lists cut, floats
rounded, the keys that matter (`ok`, `error`, `code`, `hint`) first.

In [ ]:
helper = al.Agent(al.connect("scripted"), WORK, verbose=False)
answer = helper.call("validate", model="model.json")
print("validate -> runnable:", answer["runnable"], "| findings:", answer["checks"]["findings"])

In [ ]:
# a mistake on purpose: an unknown option is refused BEFORE anything runs, and the answer says what is allowed
print(al.compact(helper.call("edit", model="model.json", rate=300, output="x.json"), 400))
# and a wrong path inside the model: kusim's own error is located and has hints
print(al.compact(helper.call("edit", model="model.json", set=["producers.0.tarjet=2000"], output="x.json"), 500))

In [ ]:
helper.call("edit", model="model.json", set=["producers.0.target=2300"], output="by_hand.json")
answer = helper.call("run", model="by_hand.json", out="runs/by_hand")
p1 = answer["wells"]["P1"]
print(f"healthy: {answer['healthy']} | P1 after {answer['final_time']:g} days: oil rate {p1['oil_rate']:.1f} STB/d, cumulative oil {p1['cum_oil']:,.0f} STB")

Now the same thing as a **loop**. This is the whole idea in about 15 lines (the real `Agent.run` adds the checks, the approval, the trace and the step limit):

In [ ]:
def tiny_agent(llm, task, helper, max_steps=8):
    messages = [{"role": "system", "content": al.SYSTEM_PROMPT}, {"role": "user", "content": task}]
    for _ in range(max_steps):
        reply = llm.chat(messages, helper.tools)                      # 1. the model looks and decides
        if not reply.tool_calls:
            return reply.text                                          #    no tool wanted: it answers
        messages.append({"role": "assistant", "content": reply.text or None, "tool_calls": [
            {"id": c.id, "type": "function", "function": {"name": c.name, "arguments": json.dumps(c.arguments)}} for c in reply.tool_calls]})
        for call in reply.tool_calls:
            result = helper.call(call.name, **call.arguments)         # 2. the program acts
            messages.append({"role": "tool", "tool_call_id": call.id, "content": al.compact(result)})   # 3. and tells the model
    return "(step limit)"


plan = [{"tool_calls": [{"name": "validate", "arguments": {"model": "by_hand.json"}}]},
        lambda messages: {"text": "validate said runnable = " + str(sc.results(messages)[-1]["runnable"])}]
print(tiny_agent(al.ScriptedLLM(plan), "Is by_hand.json fine to run?", helper))

## 4. Task A: a what-if

> *The model is model.json. What happens to the cumulative oil of producer P1 at the end of the run if the producer's bottom-hole pressure is lowered from 2500 to 2000 psi? Keep model.json as it is; run both cases and compare.*

**Predict first.** A lower bottom-hole pressure means more drawdown. Will cumulative oil go up or down, and by roughly how much (a few
percent or a factor of two)? Write your guess down, then run the agent.

*A local model takes a few minutes for this task. The cell shows each call as it happens; a failed call is not a failure of the notebook, it is the agent being corrected.*

In [ ]:
agent = new_agent("A")
run = agent.run(sc.TASKS["A"])

Read the **trace**: one line per tool call. A good agent validates before it runs, writes a new file instead of overwriting, and says what it did *not* check.

In [ ]:
def show_trace(run):
    rows = []
    for event in al.read_trace(run.trace):
        if event["event"] == "tool":
            args = ", ".join(f"{k}={v}" for k, v in event["arguments"].items())
            rows.append((event["step"], event["tool"], args[:90], event["ok"], (json.loads(event["result"]).get("code") or "") if not event["ok"] else ""))
    return pd.DataFrame(rows, columns=["step", "tool", "arguments", "ok", "error code"])


show_trace(run)

In [ ]:
# do not take the agent's word for it: read the two result files yourself
cum = lambda name: json.loads((WORK / "runs" / name / "summary.json").read_text())["wells"]["P1"]["cum_oil"]
base, low = cum("base"), cum("bhp2000")
print(f"cumulative oil of P1: {base:,.0f} STB (2500 psi) -> {low:,.0f} STB (2000 psi): {100 * (low - base) / base:+.1f} %")
print("tokens used:", run.tokens, "| steps:", run.steps)

**Questions.** Did the agent's numbers match yours? Was your prediction right? Which of its calls would you have skipped, and which
would you have added? (An agent that does not call `validate` is not wrong here, but it would be on Task B.)

## 5. Task B: fix the run

> *Run model_b.json and tell me the final oil rate and water cut of P1. I want numbers I can trust: if anything is wrong with the run or the model, fix it in a copy and say what you changed.*

`model_b.json` has a time step of 25 days and a high injection rate. IMPES (the explicit saturation update) is only conditionally stable, so
the run will **finish** but be wrong. Watch how the agent finds out, and how it reacts when one of its own calls is refused.

*Expect 2 to 5 minutes with `qwen3:8b`. Small models often skip a step or invent an argument; watch whether it uses the `suggested_dt` from `validate`.*

In [ ]:
agent = new_agent("B")
run = agent.run(sc.TASKS["B"])
show_trace(run)

In [ ]:
as_given = json.loads((WORK / "runs/b_asgiven/summary.json").read_text())
fixed = json.loads((WORK / "runs/b_fixed/summary.json").read_text())
print("as given: healthy =", as_given["healthy"], "| largest saturation correction:", round(as_given["diagnostics"]["max_saturation_clip"], 1),
      "| validate said:", as_given["checks"]["findings"][0]["code"])
print("fixed   : healthy =", fixed["healthy"], "| largest saturation correction:", round(fixed["diagnostics"]["max_saturation_clip"], 4))
print(json.dumps(helper.call("diff", a="model_b.json", b="model_b_fixed.json"))[:300])

Notice why this works: a run can end with `"ok": true` and still be wrong, so kusim reports `healthy` and `diagnostics`, `validate` gives a
`suggested_dt`, and the agent was told (in `SYSTEM_PROMPT`) to check them. Errors that say *what* is wrong and *how to fix it*
are what let a program recover without a human.

## 6. Task C: from field files to a conclusion

> *The folder field/ has what the field team sent: the deck waterflood.DATA, history.csv (measured data, day 0 is the deck's START, 1 January 2020) and later.csv (data after the history). Import the deck and the history, check how well the deck model reproduces the history, calibrate the permeability with ES-MDA (parameters.json and prior.json are in field/), and report. Use the later data to check the calibrated model. Write your conclusion in report.md.*

First look at what the agent is given: an export in the field's names and units (bar, m3/d, percent), with blank cells.

*This is the longest task (about a dozen calls and a 40-member ensemble, which also asks for your approval). With a local 8B model it may take 10 minutes or more, and it may not finish within the step limit; if so, read the trace to see where it got stuck, or run it with the scripted plan by setting `LIVE = False` in section 2.*

In [ ]:
print((WORK / "field" / "waterflood.DATA").read_text().splitlines()[0])
pd.read_csv(WORK / "field" / "history.csv").head(6)

The agent will import the deck (`import_deck`), then the data (`import_history`: units, well names, sigma, 30-day intervals), run
`misfit`, then **`ensemble`**, which asks for your **approval** (type `y` with a live model; the scripted run approves automatically and says so).
The ensemble runs 40 simulations in 4 parallel processes, about 30 seconds (the model's own thinking takes far longer).

In [ ]:
agent = new_agent("C", max_steps=20)
run = agent.run(sc.TASKS["C"])

In [ ]:
show_trace(run)

In [ ]:
print((WORK / "report.md").read_text())

## 7. Audit the agent

The agent has written a report with a conclusion. Do not accept it because it sounds sure. Check four things.

**(a) Did it use data it was not told about?** Compare the misfit (RMS of residual / sigma; 1 means "within the noise") on the data used for
the calibration (*history*) and on the data it was **not** given (*later*), for the deck model and for the calibrated model.

In [ ]:
events = al.read_trace(run.trace)


def answers(tool):
    out = []
    for e in events:
        if e["event"] == "tool" and e["tool"] == tool and e["ok"]:
            try:
                out.append((e["arguments"], json.loads(e["result"])))
            except ValueError:
                pass
    return out


rows = [(a["model"], a["observations"], r["rms"]) for a, r in answers("misfit")]
table = pd.DataFrame(rows, columns=["model", "data", "misfit (RMS / sigma)"])
table

**(b) Did it read the warnings?** The agent's answers carry `dropped` and `defaults_used` for the data and `healthy` for the calibration.
The "blank value" rows are mostly cells that are empty by nature (an injector has no oil rate; a producer no injection rate) and two gauges that failed:
kusim counts them and shows the first rows so that you can check. Silent defaults would be the problem; here sigma was given in the request.

In [ ]:
imp = answers("import_history")[0][1]
print("rows dropped when importing:", {k: v["count"] for k, v in imp["dropped"].items()})
print("defaults used for sigma:", imp["defaults_used"])
ens = answers("ensemble")[0][1]
print("ensemble healthy:", ens["healthy"], "| warnings:", ens["warnings"])
print("misfit by pass:", ens["metrics"]["rms_by_pass"])

**(c) Is the permeability right?** In real life nobody knows. Here we do: the data come from a hidden model. How far is the
calibrated permeability from the truth, compared with the deck's model we started from? (RMS error of ln kx; smaller is better.)

In [ ]:
score = case.score_against_truth(WORK / "calibration" / "posterior_mean.json", folders["hidden"])
print(score)

truth = kusim.load_model(folders["hidden"] / "truth_model.json")
start = kusim.load_model(WORK / "model_c.json")
calibrated = kusim.load_model(WORK / "calibration" / "posterior_mean.json")
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)
shape = (truth.grid.nz, truth.grid.ny, truth.grid.nx)
for ax, (name, m) in zip(axes, [("hidden truth", truth), ("deck model (start)", start), ("calibrated", calibrated)]):
    image = ax.imshow(np.log10(m.kx.reshape(shape)[1]), origin="lower", vmin=1.5, vmax=3.3, cmap="viridis")
    ax.set_title(name)
fig.colorbar(image, ax=axes, label="log10 kx (md), middle layer")
plt.show()

**(d) Rewrite the claim.** A good match to the history is not proof that the model is right: there are 3 wells and 144 blocks. Here is a
claim that follows from the numbers above; compare it with the agent's `report.md`.

In [ ]:
def misfit_of(model_part, data_part):
    sel = table[table.model.str.contains(model_part) & table.data.str.contains(data_part)]["misfit (RMS / sigma)"]
    return sel.iloc[-1]


print(f"""The calibrated model reproduces the history (misfit {misfit_of('calibration', 'history'):.1f}, the deck model had {misfit_of('model_c', 'history'):.0f}).
On the later data, which were not used, its misfit is {misfit_of('calibration', 'later'):.1f} (the deck model had {misfit_of('model_c', 'later'):.0f}): better than the start, but
far from 1, so the model is not yet a reliable forecaster. The error of ln kx against the truth went from
{score['deck_model']} to {score['calibrated']}: the data helped only partly. Next: more members, a different sigma, check
the residuals by well, and more data before this model is used for a decision.""")

**Questions.** (1) Which sentence of the agent's report is not supported by your table? (2) Which tool answer would have told the agent to
be careful, and did the system prompt tell it to look? (3) Change `SYSTEM_PROMPT` (for example add "report the misfit on the later data
next to the history misfit") and run Task C again with a live model: does the report change?

## 8. The same agent over HTTP

Everything above ran `python -m kusim ...` in a folder. The same commands are also available over **HTTP** (`kusim serve`), so an agent can drive a simulator
that runs on another machine or in a container. Only the transport changes; the tools, the checks and the answers are the same.
This needs `pip install kusim[api]` (FastAPI and uvicorn).

In [ ]:
if importlib.util.find_spec("fastapi") and importlib.util.find_spec("uvicorn"):
    server = al.HttpTransport.start(al.Workspace(WORK), port=8765)          # starts `kusim serve` on this workspace folder
    try:
        plan = [{"tool_calls": [{"name": "run", "arguments": {"model": "model.json", "out": "runs/http_base"}}]}, {"text": "done"}]
        http_agent = al.Agent(al.connect("scripted", script=plan), WORK, transport=server, verbose=False, trace="trace_http.jsonl")
        http_agent.run("run model.json")
    finally:
        server.close()
    over_http = json.loads((WORK / "runs/http_base/summary.json").read_text())["wells"]["P1"]["cum_oil"]
    print(f"cumulative oil of P1: over HTTP {over_http:,.0f} STB, over the command line {base:,.0f} STB: identical = {over_http == base}")
else:
    print("FastAPI is not installed in this environment: pip install kusim[api] (or: uv add fastapi uvicorn), then run this cell again.")

## 9. Compare local models (optional)

Run Task A three times on each extra model you list. Runs differ (the model is not deterministic) and small models fail differently from larger ones: count successes, steps and time
rather than judging one run. Each model takes minutes per run; leave the list empty to skip.

In [ ]:
COMPARE = []        # for example ["llama3.1:8b", "mistral-nemo"]: models you have already pulled with ollama pull
rows = []
for base in COMPARE:
    try:
        name = al.ollama_prepare(base, base.split(":")[0] + "-16k")
    except al.LLMError as problem:
        print(problem)
        continue
    for attempt in range(3):
        started = time.time()
        a = al.Agent(al.connect("ollama", model=name), WORK, approve=al.deny_all, verbose=False, trace=f"trace_cmp_{name}.jsonl", max_steps=10)
        r = a.run(sc.TASKS["A"])
        rows.append((base, attempt, r.stopped, r.steps, sum(not c["ok"] for c in r.calls), round(time.time() - started)))
if rows:
    display(pd.DataFrame(rows, columns=["model", "run", "stopped", "tool calls", "failed calls", "seconds"]))
else:
    print("no extra models listed: skipped")

## 10. Add your own tool

kusim's commands are the tools a *simulator* user needs. A real project has small helpers of its own. Giving an agent a new tool is a description plus a
Python function. Example: field data arrive in bar and m3/d, kusim wants psi and STB/day, and a model that converts in its head makes arithmetic mistakes.

In [ ]:
FACTORS = {"bar": 14.5037738, "mpa": 145.037738, "kpa": 0.145037738, "atm": 14.6959488, "m3/d": 6.28981077}


def to_oilfield(value, unit):
    if unit.lower() not in FACTORS:
        return {"ok": False, "code": "unknown_unit", "error": f"unit {unit!r} is not known", "known": sorted(FACTORS)}
    return {"ok": True, "value": value * FACTORS[unit.lower()], "unit": "psi" if unit.lower() != "m3/d" else "STB/d"}


spec = {"name": "to_oilfield", "description": "Convert a pressure (bar, MPa, kPa, atm) to psi or a rate in m3/d to STB/d.",
        "parameters": {"type": "object", "properties": {"value": {"type": "number"}, "unit": {"type": "string"}}, "required": ["value", "unit"]}}

plan = [{"text": "The gauge reads 196.9 bar; kusim needs psi.", "tool_calls": [{"name": "to_oilfield", "arguments": {"value": 196.9, "unit": "bar"}}]},
        lambda m: {"text": f"196.9 bar = {sc.results(m)[-1]['value']:.0f} psi."}]
agent = al.Agent(al.connect("scripted", script=plan), WORK, extra_tools={"to_oilfield": (spec, to_oilfield)}, trace="trace_units.jsonl")
agent.run("Convert 196.9 bar to psi.");

## 11. What made this possible

An agent is only as good as the system it drives. These parts of kusim are what made the three tasks work:

| Part of kusim | What it gave the agent |
|---|---|
| One command per capability, one JSON answer each | tools generated from the command line; answers a program can read |
| `kusim schema`, strict readers, errors with `path`, `code`, `did_you_mean` | recovery from mistakes without a human |
| `validate`, `healthy`, `diagnostics`, `dropped`, `defaults_used` | things to check before believing a number |
| Guardrail limits that the agent may lower but not raise | an agent that cannot start a week-long run by accident |
| `import-deck`, `import-history` | field-style input: an ECLIPSE-style deck and a messy table of measurements |
| `ensemble --workers` | parallel members: a calibration in seconds, not hours |
| `kusim serve` | the same commands over HTTP, for a remote simulator |
| `AGENTS.md` | the instructions written once, for any agent |

## 12. Try it yourself

1. **Prediction.** For Task A, predict the result for a bottom-hole pressure of 1500 psi, then ask the agent (`new_agent` with your own request: `new_agent("A").run("...")`).
2. **Break something.** Add a mistake to `model.json` (for example `"porossity"`), ask the agent to run it, and read how the error's `did_you_mean` is used.
3. **A tighter system prompt.** Edit `SYSTEM_PROMPT` in `agent_lab.py` so that every answer must list what was not checked. Compare reports.
4. **A new tool.** Write a tool that returns the water cut of a producer at a given day from `results.json` (`kusim run ... --out` writes it), give it to the agent, and ask for the breakthrough day.
5. **Another model.** Pull another model that supports tools, set `MODEL_BASE` in section 2 and run Tasks A and B again. Count steps and failed calls.
6. **Gradients.** Ask a live agent to find out which parameter the history is most sensitive to (`gradient` is a tool; the files `field/parameters.json` and `obs_history.json` are ready).

**Where next.** Wrap the tools in an MCP server so that chat applications can use them; give the agent a plotting tool; keep a human at the gates that matter (units, sigma, parameters, the final claim).

## Switching models

| You want | Do |
|---|---|
| a bigger local model | pull it (`ollama pull <name>`), set `MODEL_BASE` and `MODEL` in section 2 |
| a hosted model (Anthropic, OpenAI, Mistral, Groq, Gemini ...) | open `tutorial_agentic_workflow_frontier.ipynb`, or set `KUSIM_LLM_PROVIDER` and a key (see `docs/agent-tutorial.md`) |
| no model at all | the notebook already falls back to the scripted plans when no local model is ready |